# Choose Fisher accuracy using one default-accuracy calculation

This notebook follows the numbered Cobaya examples and calls the shared
CosmoLike Fisher functions directly, as in LSST Y1. The expensive starting
calculation differentiates every listed parameter once at each of two step
sizes. Its parameter correlations identify which derivative columns matter
most for the marginalized $(w_0,w_a)$ figure of merit. Subsequent screens
recompute only those columns. A final full calculation checks the settings
selected by that screen.

Choose NLA or TATT with the numbered YAML below. The fixed simulation covariance,
source reconstruction, IA normalization and supplied bispectrum remain those of
the YAML. Opening dark energy is an explicit change for this diagnostic.
**These cells have not been rerun during the refactor.** Running them evaluates
real theory models; no saved historical forecast is substituted automatically.

These examples use the unmasked, uncompressed row order of numbered examples 1
and 2. A different dataset mask or compression requires matching row labels
before using the two-/three-point split shown here.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import os
# BLAS must not start another worker team inside the native OpenMP calculation.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"

from copy import deepcopy
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from scipy.linalg import solve_triangular
from cobaya.model import get_model
from cobaya.yaml import yaml_load_file, yaml_dump_file

project = Path.cwd().resolve()
while not (project / "EXAMPLE_EVALUATE1.yaml").exists():
    if project == project.parent:
        raise FileNotFoundError("Open this notebook from the DES project directory.")
    project = project.parent
cocoa = project.parents[1]
sys.path.insert(0, str(project / "interface"))
sys.path.insert(0, str(cocoa / "external_modules/code/cosmolike_core"))
import cosmolike_des_3pcf_interface as ci
import cosmolike_notebook_utils.fisher as cnu

# Numbered Cocoa YAMLs resolve package and dataset paths from the Cocoa directory.
os.chdir(cocoa)

In [ ]:
# Figure style of the whole notebook: the shared plotting functions never change
# rcParams, so these lines set every font, tick and grid default.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'

## 1. State the parameter space and its physical steps

In [ ]:
# Example 1 is NLA; example 2 is TATT. Run this notebook separately for each.
example = "EXAMPLE_EVALUATE2.yaml"
info = yaml_load_file(str(project / example))
fiducial = dict(info["sampler"]["evaluate"]["override"])
info.pop("sampler")
info.pop("output", None)
info["likelihood"]["des_3pcf.joint_shear"]["omp_threads"] = 4

# Unlike the numbered inference example, this diagnostic varies dark energy.
# Removing the fixed values lets Cobaya pass the two stencil coordinates to CAMB.
info["params"]["w0"] = {
    "prior": {"min": -2.0, "max": -0.3},
    "ref": -1.0,
    "drop": True,
}
info["params"]["wa"] = {
    "prior": {"min": -2.0, "max": 2.0},
    "ref": 0.0,
}
fiducial["w0"] = -1.0
fiducial["wa"] = 0.0

# These are absolute physical displacements, not percentages of a fiducial.
# Small nuisance steps and larger dark-energy steps therefore have explicit units.
steps = {
    "omega_m": 0.003,
    "sigma8": 0.005,
    "omega_b": 0.0005,
    "h": 0.005,
    "ns": 0.005,
    "a1": 0.01,
    "eta1": 0.03,
    "m1": 0.001,
    "m2": 0.001,
    "m3": 0.001,
    "m4": 0.001,
    "dz1": 0.001,
    "dz2": 0.001,
    "dz3": 0.001,
    "dz4": 0.001,
    "w0": 0.02,
    "wa": 0.04,
}
# Gaussian source-calibration widths enter once. Flat prior limits do not
# supply Gaussian information; their posterior truncation is absent here.
priors = {
    "m1": 0.0091,
    "m2": 0.0078,
    "m3": 0.0076,
    "m4": 0.0076,
    "dz1": 0.018,
    "dz2": 0.015,
    "dz3": 0.011,
    "dz4": 0.017,
}

if example == "EXAMPLE_EVALUATE2.yaml":
    steps["a2"] = 0.02
    steps["eta2"] = 0.03
    steps["bta"] = 0.02

In [ ]:
def get_dv(parameters):
    """Evaluate the current numbered Cobaya model at a physical parameter point.

    Cobaya runs its declared theory blocks before the DES likelihood. The
    likelihood retains the full mean it evaluated; the native selector applies
    its data mask and compression. A Fisher displacement must reach that same
    prediction, not another CAMB or bispectrum preparation route.

    Arguments:
      parameters = dictionary of sampled physical parameters for the current model.
    Returns:
      Owned one-dimensional selected data vector, in the covariance's row order.
    Raises:
      RuntimeError if the likelihood evaluation is not finite.
    Side effects:
      Updates the current model and the process-global native cosmology.
    """
    value = model.loglike(
        params_values=parameters,
        return_derived=False,
        cached=False,
    )
    if not np.isfinite(value):
        raise RuntimeError("A Fisher point has nonfinite likelihood; inspect that point.")
    return np.array(
        object=ci.select_joint_data_vector(prediction=likelihood.last_datavector),
        dtype=np.float64,
        copy=True,
    )

## 2. Compute the full base at default accuracy

The derivative columns are physical derivatives, with one column for each
name in `steps`. The five-point rules at $h$ and $h/2$ expose step sensitivity.
Numerical integrations can still share a smooth error at both steps, so this
agreement alone does not prove convergence.

The shared mean cache retains identical physical points **inside this single
fixed-accuracy calculation**. It avoids repeating the common center and shared
stencil nodes. A new model and a new cache are constructed for every accuracy
case; a vector at one sampling cannot serve as a vector at another sampling.

In [ ]:
model = get_model(info_or_yaml_or_file=info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
try:
    covariance = np.array(object=ci.get_joint_covariance(), copy=True)
    predict = cnu.ExactMeanCache(predict=get_dv)
    names, center, dh, dh2 = cnu.differentiate(
        predict=predict,
        parameters=fiducial,
        steps=steps,
    )
    base = cnu.fisher_matrix(
        derivatives=dh2,
        names=names,
        covariance=covariance,
        priors=priors,
    )
    base_h = cnu.fisher_matrix(
        derivatives=dh,
        names=names,
        covariance=covariance,
        priors=priors,
    )
    iw0 = names.index("w0")
    iwa = names.index("wa")
    invcov = np.linalg.solve(covariance, np.eye(covariance.shape[0]))
    eps = np.asarray(list(steps.values()), dtype=float) / 2.0
    weights = cnu.get_fom_noise_weights(F=base, i=iw0, j=iwa, eps=eps)
    # Absolute offsets avoid rescaling the stencil by a signed or zero fiducial.
    offset_dv = cnu.make_offset_predictor(
        predict=predict,
        reference=fiducial,
        names=names,
    )
    baseline_a = np.zeros(len(names))
    baseline_sigma = np.zeros(len(names))
    for index in range(len(names)):
        noise = cnu.get_stencil_noise(
            dv=offset_dv,
            index=index,
            h=eps[index],
            CV=np.zeros(len(names)),
            invcov=invcov,
        )
        baseline_a[index] = noise["a"]
        baseline_sigma[index] = noise["sigma"]
    print("FoM at h:", cnu.get_FoM(i=iw0, j=iwa, F=base_h))
    print("FoM at h/2:", cnu.get_FoM(i=iw0, j=iwa, F=base))
finally:
    model.close()

## 3. Estimate the positive information from derivative noise

Five equally spaced predictions form a fourth difference:

$$\Delta^4m=m(-2h)-4m(-h)+6m(0)-4m(h)+m(2h).$$

Its squared covariance-weighted norm divided by 70 estimates the evaluation
noise variance under independent, equal-variance numerical fluctuations. The
factor 70 is the sum of the squared coefficients. This estimate supplies the
false derivative information $a_j=(130/144)\hat\sigma_j^2/h_j^2$.

The full base Fisher matrix determines how that column affects the dark-energy
ellipse after marginalizing all listed parameters. The shared weights give
$\delta\ln\mathrm{FoM}\simeq\tfrac12\sum_jR_j^2\sigma_j^2a_j$.
The next cell selects columns by their **measured baseline contribution**, then
prints the omitted contribution. A small baseline contribution is not a bound
on its value at another accuracy. Smooth fourth derivatives and correlated
errors also violate the independent-noise interpretation.

In [ ]:
# R2 times sigma2 carries parameter units squared; multiplying by a
# produces a dimensionless contribution to the logarithmic FoM shift.
contributions = (weights["R2"] * weights["sigma2"]) * baseline_a
order = np.argsort(contributions)[::-1]
selected = []
accumulated = 0.0
total = np.sum(contributions)
for index in order:
    selected.append(int(index))
    accumulated += contributions[index]
    if total > 0 and accumulated >= 0.95 * total:
        break
# Exact zeros select every column rather than interpreting a zero audit as proof.
print("Varied parameters:", names)
print("Noise columns:", [names[index] for index in selected])
print("Omitted baseline dlnFoM:", 0.5 * (total - accumulated))
for name, contribution in zip(names, contributions):
    print(name, "baseline dlnFoM contribution:", 0.5 * contribution)

## 4. Change one numerical setting at a time

A new Cobaya model is initialized for each setting, so its native grid sizes
and theory requirements are rebuilt together. The three-point sampling factor
and the ordinary two-point factor are different controls; this list states
which one changes. CAMB's integration settings and the supplied bispectrum's
sampling are not silently multiplied by a three-point control.

The diagonal addition is applied to the **same** base Fisher matrix at every
setting. Both its linearized FoM shift and exact matrix ratio remain estimates
of the noise model. They are not corrected scientific FoMs.

In [ ]:
# Each entry edits an ordinary likelihood setting from joint_shear.yaml.
# Add separate theory settings when investigating CAMB or bispectrum sampling.
candidates = {
    "default": {},
    "source_2": {"threepcf_accuracy": {"source_boost": 2.0}},
    "chi_2": {"threepcf_accuracy": {"chi_boost": 2.0}},
    "triangle_1p5": {"threepcf_accuracy": {"triangle_boost": 1.5}},
    "multipole_1p5": {"threepcf_accuracy": {"multipole_boost": 1.5}},
    "core_two_point_1p5": {"accuracyboost": 1.5},
    "sampling_2": {"threepcf_accuracy": {"accuracy_boost": 2.0}},
}
screen = {}
for label, changes in candidates.items():
    candidate_info = deepcopy(info)
    candidate_info["likelihood"]["des_3pcf.joint_shear"].update(changes)
    model = get_model(info_or_yaml_or_file=candidate_info)
    likelihood = model.likelihood["des_3pcf.joint_shear"]
    try:
        # Never carry a retained mean across changes in numerical resolution.
        predict = cnu.ExactMeanCache(predict=get_dv)
        offset_dv = cnu.make_offset_predictor(
            predict=predict,
            reference=fiducial,
            names=names,
        )
        addition = np.zeros(len(names))
        for index in selected:
            noise = cnu.get_stencil_noise(
                dv=offset_dv,
                index=index,
                h=eps[index],
                CV=np.zeros(len(names)),
                invcov=invcov,
            )
            addition[index] = noise["a"]
        estimate = cnu.predict_fom_noise(F=base, i=iw0, j=iwa, a=addition)
        screen[label] = estimate
        print(label, "estimated dlnFoM:", estimate["dlnFoM"], "ratio:", estimate["ratio"])
    finally:
        model.close()

## 5. Confirm with full derivatives and smaller steps

Choose one candidate from the screen and recalculate every column. Compare
actual marginalized FoMs at $h$ and $h/2$, as well as covariance-weighted column
changes. Repeat for another candidate if needed. Do not subtract a predicted
noise shift from the scientific FoM.

For a cusp, step halving can leave the false-information estimate nearly
constant; for a smooth fourth-order term, the fourth difference scales as
$h^4$. Shifted stencil centers, one-sided slopes and more step sizes can help
identify those behaviors. These are diagnostics of numerical derivatives,
not additional physical measurement covariance.

In [ ]:
# This cell is a real full forecast when executed. Select the setting after
# inspecting the table above; the name below is only an explicit example.
chosen = "sampling_2"
refined_info = deepcopy(info)
refined_info["likelihood"]["des_3pcf.joint_shear"].update(candidates[chosen])
model = get_model(info_or_yaml_or_file=refined_info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
try:
    predict = cnu.ExactMeanCache(predict=get_dv)
    names, refined_center, refined_h, refined_h2 = cnu.differentiate(
        predict=predict,
        parameters=fiducial,
        steps=steps,
    )
    refined_fishers = []
    for derivative in (refined_h, refined_h2):
        fisher = cnu.fisher_matrix(
            derivatives=derivative,
            names=names,
            covariance=covariance,
            priors=priors,
        )
        refined_fishers.append(fisher)
        print(chosen, "actual FoM:", cnu.get_FoM(i=iw0, j=iwa, F=fisher))
    factor = np.linalg.cholesky(covariance)
    change = solve_triangular(a=factor, b=refined_h2-dh2, lower=True)
    reference = solve_triangular(a=factor, b=refined_h2, lower=True)
    for index, name in enumerate(names):
        ratio = np.linalg.norm(change[:, index]) / np.linalg.norm(reference[:, index])
        print(name, "accuracy change in derivative [%]:", 100.0 * ratio)
    directory = project / "chains/fisher_accuracy"
    directory.mkdir(parents=True, exist_ok=True)
    yaml_dump_file(str(directory / "base.yaml"), info, error_if_exists=False)
    yaml_dump_file(str(directory / "refined.yaml"), refined_info, error_if_exists=False)
    np.savez(
        file=directory / "derivatives.npz",
        names=np.asarray(names),
        steps=np.asarray(list(steps.values())),
        covariance=covariance,
        base_derivatives_h=dh,
        base_derivatives_h2=dh2,
        refined_derivatives_h=refined_h,
        refined_derivatives_h2=refined_h2,
        base_fisher=base,
        refined_fisher_h=refined_fishers[0],
        refined_fisher_h2=refined_fishers[1],
        baseline_noise=baseline_a,
        selected_columns=np.asarray(selected),
    )
finally:
    model.close()